# Customer Churn Analytics

Análise exploratória, segmentação de risco e recomendações de retenção. Os dados são sintéticos e não representam clientes reais.

## 1. Preparação do ambiente

In [ ]:
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data' / 'processed' / 'customer_churn_clean.csv'
df = pd.read_csv(DATA)
df.head()

## 2. Qualidade e visão geral

In [ ]:
quality = pd.DataFrame({
    'tipo': df.dtypes.astype(str),
    'nulos': df.isna().sum(),
    'unicos': df.nunique(),
})
quality

In [ ]:
kpis = {
    'clientes': len(df),
    'churn_rate': df['churn'].mean(),
    'mrr_ativo': df.loc[df.churn.eq(0), 'monthly_charges'].sum(),
    'receita_anual_em_risco': df.loc[df.churn.eq(1), 'annual_revenue'].sum(),
}
kpis

## 3. Diagnóstico dos principais segmentos

In [ ]:
segment = (df.groupby(['contract', 'plan'], observed=True)
             .agg(clientes=('customer_id', 'count'),
                  churn_rate=('churn', 'mean'),
                  receita_em_risco=('revenue_at_risk', 'sum'))
             .sort_values('churn_rate', ascending=False))
segment.head(10)

In [ ]:
order = df.groupby('contract')['churn'].mean().sort_values(ascending=False).index
sns.barplot(data=df, x='contract', y='churn', order=order, errorbar=None, color='#2563EB')
plt.title('Taxa de churn por contrato')
plt.ylabel('Churn')
plt.xlabel('')
plt.show()

## 4. Lista de clientes para ação preventiva

In [ ]:
priority = (df.query("churn == 0 and risk_segment == 'Alto'")
              .sort_values(['risk_score', 'annual_revenue'], ascending=False))
priority[['customer_id', 'plan', 'contract', 'nps', 'risk_score', 'annual_revenue']].head(20)

## 5. Recomendações

1. Priorizar clientes mensais de alto risco nos primeiros seis meses.
2. Criar acionamento após o terceiro chamado de suporte.
3. Testar incentivo ao débito automático e migração anual.
4. Mensurar uplift e ROI com grupo de controle.